# Global Roadkill — Final Documentary Analysis




In [ ]:



import os
import calendar
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


HIGHLIGHT = '#B23A48'
MUTED = '#B9AFA3'
DARK = '#3A3A3A'
LIGHT_GRID = '#EDEAE5'
FONT = dict(family='Georgia, serif', size=15, color=DARK)

PALETTE = {
    'Mammalia': '#B23A48',
    'Aves': '#4C7B8C',
    'Reptilia': '#6B8E63',
    'Amphibia': '#D4A24C'
}


if not os.path.exists('Global Roadkill data.csv'):
    from google.colab import files
    print('Upload Global Roadkill data.csv')
    files.upload()


df = pd.read_csv('Global Roadkill data.csv', low_memory=False)

keep_cols = [
    'occurrenceID', 'country', 'continent', 'locality',
    'decimalLatitude', 'decimalLongitude',
    'class', 'order', 'family', 'scientificName', 'vernacularName',
    'iucnStatus', 'numberOfRoadkill',
    'year', 'month', 'day',
    'roadType', 'roadLength', 'surveyType'
]

existing_cols = [c for c in keep_cols if c in df.columns]
df = df[existing_cols].copy()


if 'scientificName' in df.columns:
    df = df.dropna(subset=['scientificName', 'decimalLatitude', 'decimalLongitude'])

if 'numberOfRoadkill' in df.columns:
    df['numberOfRoadkill'] = pd.to_numeric(df['numberOfRoadkill'], errors='coerce').fillna(1)
    df.loc[df['numberOfRoadkill'] < 0, 'numberOfRoadkill'] = np.nan
    df['numberOfRoadkill'] = df['numberOfRoadkill'].fillna(1)

if 'vernacularName' in df.columns:
    df['vernacularName'] = df['vernacularName'].fillna(df['scientificName'])

if 'iucnStatus' in df.columns:
    df['iucnStatus'] = df['iucnStatus'].fillna('NE').astype(str).str.strip().str.upper()

if 'roadType' in df.columns:
    df['roadType'] = df['roadType'].fillna('Unknown')

if 'occurrenceID' in df.columns:
    df.drop_duplicates(subset='occurrenceID', inplace=True)


clean_path = 'global_roadkill_clean.csv'
df.to_csv(clean_path, index=False)

print(f'Cleaned dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Cleaned CSV saved as: {clean_path}')


In [ ]:
!pip uninstall kaleido -y -q
!pip install kaleido==0.2.1 -q

# INSIGHT 1 — What kinds of animals are being hit?

**Documentary purpose:** establish the scale of the problem and show that mammals dominate the recorded roadkill dataset.

In [ ]:

class_totals = df.groupby('class', as_index=False)['numberOfRoadkill'].sum().sort_values('numberOfRoadkill', ascending=False)
total = class_totals['numberOfRoadkill'].sum()
class_totals['pct'] = (class_totals['numberOfRoadkill']/total*100).round(1)
avg = class_totals['numberOfRoadkill'].mean()

icons = {'Mammalia':'🦌','Amphibia':'🐸','Reptilia':'🐍','Aves':'🦅'}
HIGHLIGHT, MUTED = '#B23A48', '#B9AFA3'
colors = [HIGHLIGHT if i==0 else MUTED for i in range(len(class_totals))]

fig1 = go.Figure()

fig1.add_trace(go.Bar(
    x=class_totals['class'], y=class_totals['numberOfRoadkill'],
    marker_color=colors, marker_line_width=0,
    text=class_totals.apply(lambda r: f"{r['numberOfRoadkill']:,}<br>({r['pct']}%)", axis=1),
    textposition='outside',
    textfont=dict(size=14, family='Georgia, serif', color='#3A3A3A'),
))

fig1.add_hline(y=avg, line_dash='dot', line_color='#8A8A8A', line_width=1.5,
              annotation_text=f'Average across classes: {avg:,.0f}',
              annotation_position='top left',
              annotation_font=dict(size=11, color='#8A8A8A', family='Georgia, serif'))

for i, row in class_totals.iterrows():
    fig1.add_annotation(x=row['class'], y=row['numberOfRoadkill']*0.5,
                        text=icons.get(row['class'],''), showarrow=False,
                        font=dict(size=34))

fig1.add_annotation(
    x=0, y=class_totals['numberOfRoadkill'].max()*1.18,
    xref='x', yref='y', text="Within Mammalia, one species — the European Roe Deer —<br>accounts for over a third of the total on its own",
    showarrow=True, arrowhead=2, ax=90, ay=-40,
    font=dict(size=11.5, color='#7A1F2B', family='Georgia, serif'),
    bgcolor='#FBF3F0', bordercolor='#B23A48', borderwidth=1, borderpad=6,
)

fig1.update_layout(
    template='plotly_white',
    font=dict(family='Georgia, serif', size=15, color='#3A3A3A'),
    title=dict(text='<b>Roadkill by Animal Class</b><br><span style="font-size:13px;color:#8A8A8A">Mammals account for more than half of all recorded roadkill worldwide — nearly 2.5x the average</span>',
               x=0.02, xanchor='left'),
    yaxis=dict(title='Animals killed', tickformat=',', showgrid=True, gridcolor='#EDEAE5', zeroline=False,
               range=[0, class_totals['numberOfRoadkill'].max()*1.35]),
    xaxis=dict(title=''),
    plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(t=100, b=80, l=70, r=40),
    bargap=0.35, showlegend=False,
)
fig1.add_annotation(text='Source: Grilo et al. 2024/2025, Global Roadkill Data',
                    xref='paper', yref='paper', x=0, y=-0.16, showarrow=False,
                    font=dict(size=11, color='#A0A0A0'), xanchor='left')

fig1.write_image('/content/chart1_class.png', scale=2, width=1600, height=900)
fig1.show()

# INSIGHT 2 — Who are the most common victims?

**Documentary purpose:** move from broad animal groups to named species. The standout is the European Roe Deer.

In [ ]:



icons = {'Mammalia':'🦌','Amphibia':'🐸','Reptilia':'🐍','Aves':'🦅'}

top_species = df.groupby(['vernacularName','class'], as_index=False)['numberOfRoadkill'].sum().sort_values('numberOfRoadkill', ascending=False).head(10)
top_species = top_species.sort_values('numberOfRoadkill')
top_species['label'] = top_species.apply(lambda r: f"{icons.get(r['class'],'')} {r['vernacularName']}", axis=1)

HIGHLIGHT, MUTED = '#B23A48', '#B9AFA3'
colors = [HIGHLIGHT if i==len(top_species)-1 else MUTED for i in range(len(top_species))]

fig2 = go.Figure(go.Bar(
    x=top_species['numberOfRoadkill'], y=top_species['label'], orientation='h',
    marker_color=colors, marker_line_width=0,
    text=top_species['numberOfRoadkill'].apply(lambda v: f"{v:,}"),
    textposition='outside',
    textfont=dict(size=14, family='Georgia, serif', color='#3A3A3A'),
))

top1 = top_species.iloc[-1]
top2 = top_species.iloc[-2]
fig2.add_annotation(
    x=top1['numberOfRoadkill'], y=top_species.index.get_loc(top1.name),
    ax=top2['numberOfRoadkill'], ay=top_species.index.get_loc(top2.name)-1,
    axref='x', ayref='y', xref='x', yref='y',
    text='', showarrow=True, arrowhead=2, arrowcolor='#7A1F2B', arrowwidth=1.5,
)
fig2.add_annotation(
    x=(top1['numberOfRoadkill']+top2['numberOfRoadkill'])/2*1.15, y=8.4,
    text="Nearly 3x the<br>next species", showarrow=False,
    font=dict(size=11.5, color='#7A1F2B', family='Georgia, serif'),
    bgcolor='#FBF3F0', bordercolor='#B23A48', borderwidth=1, borderpad=5,
)

fig2.update_layout(
    template='plotly_white',
    font=dict(family='Georgia, serif', size=15, color='#3A3A3A'),
    title=dict(text='<b>Top 10 Most-Killed Species Worldwide</b><br><span style="font-size:13px;color:#8A8A8A">Icons show animal class — the European Roe Deer leads by a wide margin</span>',
               x=0.02, xanchor='left'),
    xaxis=dict(title='Animals killed', tickformat=',', showgrid=True, gridcolor='#EDEAE5', zeroline=False),
    yaxis=dict(title=''),
    plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(t=90, b=70, l=200, r=90),
    showlegend=False,
)
fig2.add_annotation(text='Source: Grilo et al. 2024/2025, Global Roadkill Data',
                    xref='paper', yref='paper', x=0, y=-0.14, showarrow=False,
                    font=dict(size=11, color='#A0A0A0'), xanchor='left')
fig2.write_image('/content/chart2_species.png', scale=2, width=1600, height=900)
fig2.show()


# INSIGHT 3 — When does roadkill peak?

**Documentary purpose:** introduce time. The chart shows the monthly pattern in the recorded dataset; it does not by itself prove that a month is intrinsically more dangerous worldwide because survey coverage and hemispheric seasons differ.

In [ ]:



month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
monthly = df.dropna(subset=['month']).groupby('month', as_index=False)['numberOfRoadkill'].sum()
monthly['month_name'] = monthly['month'].map(month_names)
monthly = monthly.sort_values('month').reset_index(drop=True)
avg3 = monthly['numberOfRoadkill'].mean()
peak_idx = monthly['numberOfRoadkill'].idxmax()
colors3 = [HIGHLIGHT if i==peak_idx else MUTED for i in monthly.index]

season_bands = [
    (-0.5, 1.5, '#DCE9F0', 'Winter'),
    (1.5, 4.5, '#E3EEDC', 'Spring'),
    (4.5, 7.5, '#FBEFD9', 'Summer'),
    (7.5, 10.5, '#F1E3D3', 'Autumn'),
    (10.5, 11.5, '#DCE9F0', ''),
]

fig3 = go.Figure()
for x0, x1, color, label in season_bands:
    fig3.add_vrect(x0=x0, x1=x1, fillcolor=color, opacity=0.5, line_width=0, layer='below',
                    annotation_text=label, annotation_position='top', annotation_font=dict(size=10.5, color='#8A8A8A'))

fig3.add_trace(go.Bar(
    x=monthly['month_name'], y=monthly['numberOfRoadkill'],
    marker_color=colors3, marker_line_width=0,
    text=monthly['numberOfRoadkill'].apply(lambda v: f"{v:,}"),
    textposition='outside',
    textfont=dict(size=12.5, family='Georgia, serif', color='#3A3A3A'),
))
fig3.add_hline(y=avg3, line_dash='dot', line_color='#8A8A8A', line_width=1.5,
              annotation_text=f'Monthly average: {avg3:,.0f}', annotation_position='bottom right',
              annotation_font=dict(size=11, color='#8A8A8A', family='Georgia, serif'))

fig3.update_layout(
    template='plotly_white',
    font=dict(family='Georgia, serif', size=15, color='#3A3A3A'),
    title=dict(text='<b>Roadkill by Month</b><br><span style="font-size:13px;color:#8A8A8A">August sees more than double the roadkill of February — peaking in late summer</span>',
               x=0.02, xanchor='left'),
    yaxis=dict(title='Animals killed', tickformat=',', showgrid=True, gridcolor='#EDEAE5', zeroline=False),
    xaxis=dict(title=''),
    plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(t=100, b=70, l=70, r=40),
    bargap=0.25, showlegend=False,
)
fig3.add_annotation(text='Source: Grilo et al. 2024/2025, Global Roadkill Data',
                    xref='paper', yref='paper', x=0, y=-0.18, showarrow=False,
                    font=dict(size=11, color='#A0A0A0'), xanchor='left')
fig3.write_image('/content/chart3_seasonality.png', scale=2, width=1600, height=900)
fig3.show()


# INSIGHT 4 — Every continent has its victims

**Documentary purpose:** take the audience around the world. The leading species change from region to region, so the problem is not represented by one global "victim."

In [ ]:



import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

CLASS_COLORS = {'Mammalia': '#B23A48', 'Amphibia': '#D4A24C', 'Reptilia': '#6B8E63', 'Aves': '#4C7B8C'}
continent_totals = df.groupby('continent')['numberOfRoadkill'].sum().sort_values(ascending=False)
continents = continent_totals.index.tolist()
n = len(continents); cols = 3; rows = (n + cols - 1)//cols

fig = make_subplots(rows=rows, cols=cols,
                     subplot_titles=[f"{c}<br><span style='font-size:11px;color:#8A8A8A'>n = {continent_totals[c]:,} records</span>" for c in continents],
                     horizontal_spacing=0.14, vertical_spacing=0.28)

for i, cont in enumerate(continents):
    r, c = divmod(i, cols)
    sub = df[df['continent']==cont]
    top3 = sub.groupby(['vernacularName','class'], as_index=False)['numberOfRoadkill'].sum().sort_values('numberOfRoadkill', ascending=False).head(3)
    total_cont = sub['numberOfRoadkill'].sum()
    top3 = top3.sort_values('numberOfRoadkill')
    top3['pct'] = (top3['numberOfRoadkill']/total_cont*100).round(0).astype(int)
    top3['label'] = top3.apply(lambda x: f"{x['numberOfRoadkill']:,} ({x['pct']}%)", axis=1)
    colors = [CLASS_COLORS.get(cl, '#999') for cl in top3['class']]
    fig.add_trace(go.Bar(x=top3['numberOfRoadkill'], y=top3['vernacularName'], orientation='h',
        marker_color=colors, marker_line_width=0, text=top3['label'], textposition='outside',
        textfont=dict(size=11, family='Georgia, serif', color='#3A3A3A'), showlegend=False,
    ), row=r+1, col=c+1)
    max_val = top3['numberOfRoadkill'].max()
    fig.update_xaxes(range=[0, max_val*1.55], showticklabels=False, row=r+1, col=c+1)
    fig.update_yaxes(tickfont=dict(size=11.5, family='Georgia, serif', color='#3A3A3A'), row=r+1, col=c+1)

for cl, color in CLASS_COLORS.items():
    fig.add_trace(go.Bar(x=[None], y=[None], marker_color=color, name=cl, showlegend=True))

fig.update_layout(
    template='plotly_white', font=dict(family='Georgia, serif', size=14, color='#3A3A3A'),
    title=dict(text="<b>Every Continent Has Its Victim</b><br><span style='font-size:13px;color:#8A8A8A'>Top 3 most-killed species per continent — the species hit hardest changes by region</span>",
               x=0.02, xanchor='left', y=0.98, yanchor='top'),
    plot_bgcolor='white', paper_bgcolor='white', height=820, width=1600,
    margin=dict(t=160, b=90, l=40, r=40),
    legend=dict(orientation='h', yanchor='bottom', y=-0.1, xanchor='center', x=0.5, title=''),
    bargap=0.4,
)
for ann in fig['layout']['annotations'][:6]:
    ann['font'] = dict(size=14, family='Georgia, serif', color='#3A3A3A')
fig.add_annotation(text='Source: Grilo et al. 2024/2025, Global Roadkill Data',
                    xref='paper', yref='paper', x=0, y=-0.14, showarrow=False,
                    font=dict(size=11, color='#A0A0A0'), xanchor='left')
fig.write_image('/content/chart_continent.png', scale=2, width=1600, height=820)
fig.show()


# INSIGHT 5 — Where is recorded roadkill concentrated?

**Documentary purpose:** replace the old point-cloud world map with a proper interactive geographic hotspot view.



In [ ]:
!pip install -q keplergl==0.4.0rc4

In [ ]:




from keplergl import KeplerGl

map_df = df[
    df['decimalLatitude'].notna() &
    df['decimalLongitude'].notna()
].copy()

map_df = map_df.rename(columns={
    'decimalLatitude': 'latitude',
    'decimalLongitude': 'longitude'
})


map_df['roadkill_weight'] = map_df['numberOfRoadkill']

map_df = map_df[[
    'latitude', 'longitude', 'roadkill_weight',
    'numberOfRoadkill', 'country', 'continent',
    'class', 'scientificName', 'vernacularName', 'iucnStatus'
]].copy()

roadkill_map = KeplerGl(height=700, width=1200)
roadkill_map.add_data(data=map_df, name='Global Roadkill')
roadkill_map


# INSIGHT 6 — Where do threatened species make up the largest share?

**Documentary purpose:** move beyond raw counts. Different continents have very different amounts of recorded data, so compare the **percentage of each continent's roadkill involving strictly threatened species**.

Strictly threatened here means **VU + EN + CR**.

In [ ]:



STRICT_THREATENED = ['VU', 'EN', 'CR']

continent_total = (
    df.groupby('continent')['numberOfRoadkill']
      .sum()
      .reset_index(name='total_roadkill')
)

continent_threatened = (
    df[df['iucnStatus'].isin(STRICT_THREATENED)]
      .groupby('continent')['numberOfRoadkill']
      .sum()
      .reset_index(name='threatened_roadkill')
)

continent_analysis = continent_total.merge(
    continent_threatened,
    on='continent',
    how='left'
)

continent_analysis['threatened_roadkill'] = continent_analysis['threatened_roadkill'].fillna(0)
continent_analysis['threatened_pct'] = (
    continent_analysis['threatened_roadkill'] /
    continent_analysis['total_roadkill'] * 100
)

continent_analysis = continent_analysis.sort_values('threatened_pct', ascending=False)

plot_data = continent_analysis.sort_values('threatened_pct')

fig6 = go.Figure(go.Bar(
    x=plot_data['threatened_pct'],
    y=plot_data['continent'],
    orientation='h',
    marker_color=HIGHLIGHT,
    marker_line_width=0,
    text=plot_data['threatened_pct'],
    texttemplate='%{text:.1f}%',
    textposition='outside',
    customdata=np.stack([
        plot_data['threatened_roadkill'],
        plot_data['total_roadkill']
    ], axis=-1),
    hovertemplate=(
        '<b>%{y}</b><br>'
        '%{x:.2f}% of recorded roadkill is VU/EN/CR<br>'
        '%{customdata[0]:,.0f} threatened / %{customdata[1]:,.0f} total<extra></extra>'
    )
))

fig6.update_layout(
    template='plotly_white',
    font=FONT,
    title=dict(
        text='<b>Where Threatened Species Make Up the Largest Share of Roadkill</b><br>'
             '<span style="font-size:13px;color:#8A8A8A">Share of recorded roadkill involving Vulnerable, Endangered or Critically Endangered species</span>',
        x=0.02, xanchor='left'
    ),
    xaxis=dict(title='Threatened roadkill (% of continent total)', ticksuffix='%', showgrid=True, gridcolor=LIGHT_GRID, zeroline=False),
    yaxis=dict(title=''),
    margin=dict(t=115, b=80, l=100, r=80),
    showlegend=False,
    plot_bgcolor='white', paper_bgcolor='white'
)

fig6.add_annotation(
    text='Source: Global Roadkill Data',
    x=0, xanchor='left', xref='paper', y=-0.17, yref='paper',
    showarrow=False, font=dict(size=11, color='#A0A0A0')
)

fig6.show()

print('\nTHREATENED ROADKILL SHARE BY CONTINENT')
print(continent_analysis.to_string(index=False, formatters={
    'total_roadkill': lambda x: f'{x:,.0f}',
    'threatened_roadkill': lambda x: f'{x:,.0f}',
    'threatened_pct': lambda x: f'{x:.2f}%'
}))


# INSIGHT 7 — How much recorded roadkill involves threatened species?

**Documentary purpose:** introduce the conservation turn. This uses the strict IUCN threatened categories only: **Vulnerable (VU), Endangered (EN), and Critically Endangered (CR)**.

**Near Threatened (NT) is intentionally excluded** because it is not formally one of the three IUCN threatened categories.

In [ ]:


strict_threatened_df = df[df['iucnStatus'].isin(STRICT_THREATENED)].copy()

status_labels = {
    'VU': 'Vulnerable',
    'EN': 'Endangered',
    'CR': 'Critically Endangered'
}

status_totals = (
    strict_threatened_df.groupby('iucnStatus', as_index=False)['numberOfRoadkill']
                        .sum()
)
status_totals['status_label'] = status_totals['iucnStatus'].map(status_labels)

status_order = ['Vulnerable', 'Endangered', 'Critically Endangered']
status_totals['status_label'] = pd.Categorical(
    status_totals['status_label'], categories=status_order, ordered=True
)
status_totals = status_totals.sort_values('status_label')

strict_total = strict_threatened_df['numberOfRoadkill'].sum()
all_total = df['numberOfRoadkill'].sum()
strict_pct = strict_total / all_total * 100

fig7 = go.Figure(go.Bar(
    x=status_totals['status_label'],
    y=status_totals['numberOfRoadkill'],
    marker_color=HIGHLIGHT,
    marker_line_width=0,
    text=status_totals['numberOfRoadkill'].apply(lambda v: f'{v:,.0f}'),
    textposition='outside'
))

fig7.update_layout(
    template='plotly_white',
    font=FONT,
    title=dict(
        text=f'<b>Roadkill Involving Strictly Threatened Species</b><br>'
             f'<span style="font-size:13px;color:#8A8A8A">{strict_pct:.1f}% of all recorded roadkill involves VU, EN or CR species</span>',
        x=0.02, xanchor='left'
    ),
    xaxis=dict(title='IUCN status'),
    yaxis=dict(title='Recorded roadkill', tickformat=',', showgrid=True, gridcolor=LIGHT_GRID, zeroline=False),
    margin=dict(t=100, b=80, l=75, r=45),
    showlegend=False,
    plot_bgcolor='white', paper_bgcolor='white'
)

fig7.add_annotation(
    text='Source: Global Roadkill Data · strict threatened categories = VU + EN + CR',
    x=0, xref='paper', y=-0.16, yref='paper',
    showarrow=False, xanchor='left', font=dict(size=11, color='#A0A0A0')
)

fig7.show()

print(f'\nStrictly threatened roadkill: {strict_total:,.0f}')
print(f'All recorded roadkill: {all_total:,.0f}')
print(f'Share involving VU/EN/CR: {strict_pct:.2f}%')


# INSIGHT 8 — Which threatened species are being recorded most often?

**Documentary purpose:** put specific, recognizable species at the center of the conservation story. Each species is shown with its IUCN status.

In [ ]:


top_threatened = (
    strict_threatened_df
    .groupby(['vernacularName', 'iucnStatus'], as_index=False)['numberOfRoadkill']
    .sum()
    .sort_values('numberOfRoadkill', ascending=False)
    .head(10)
)

plot_threatened = top_threatened.sort_values('numberOfRoadkill')

plot_threatened['label'] = (
    plot_threatened['vernacularName'] +
    ' (' + plot_threatened['iucnStatus'] + ')'
)

colors = [MUTED] * len(plot_threatened)
colors[-1] = HIGHLIGHT

fig8 = go.Figure(go.Bar(
    x=plot_threatened['numberOfRoadkill'],
    y=plot_threatened['label'],
    orientation='h',
    marker_color=colors,
    marker_line_width=0,
    text=plot_threatened['numberOfRoadkill'].apply(lambda v: f'{v:,.0f}'),
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>%{x:,.0f} recorded roadkill<extra></extra>'
))

fig8.update_layout(
    template='plotly_white',
    font=FONT,
    title=dict(
        text='<b>Top 10 Threatened Species in Roadkill Records</b><br>'
             '<span style="font-size:13px;color:#8A8A8A">Species already classified as Vulnerable, Endangered or Critically Endangered</span>',
        x=0.02, xanchor='left'
    ),
    xaxis=dict(title='Recorded roadkill', tickformat=',', showgrid=True, gridcolor=LIGHT_GRID, zeroline=False),
    yaxis=dict(title=''),
    margin=dict(t=100, b=70, l=210, r=70),
    showlegend=False,
    plot_bgcolor='white', paper_bgcolor='white'
)

fig8.add_annotation(
    text='Source: Global Roadkill Data · VU = Vulnerable, EN = Endangered, CR = Critically Endangered',
    x=0, xref='paper', y=-0.15, yref='paper',
    showarrow=False, xanchor='left', font=dict(size=10, color='#A0A0A0')
)

fig8.show()

print('\nTOP 10 THREATENED SPECIES')
for i, (_, row) in enumerate(top_threatened.iterrows(), 1):
    print(f"{i}. {row['vernacularName']} ({row['iucnStatus']}) — {row['numberOfRoadkill']:,.0f}")


# INSIGHT 9 — Where are threatened-species roadkill hotspots?

**Documentary purpose:** finish by bringing the conservation story back onto the map. This is the same geographic idea as Insight 5, but filtered to **VU + EN + CR** only.

In [ ]:


threatened_map_df = df[
    df['iucnStatus'].isin(STRICT_THREATENED) &
    df['decimalLatitude'].notna() &
    df['decimalLongitude'].notna()
].copy()

threatened_map_df = threatened_map_df.rename(columns={
    'decimalLatitude': 'latitude',
    'decimalLongitude': 'longitude'
})

threatened_map_df['roadkill_weight'] = threatened_map_df['numberOfRoadkill']

threatened_map_df = threatened_map_df[[
    'latitude', 'longitude', 'roadkill_weight',
    'numberOfRoadkill', 'country', 'continent',
    'class', 'scientificName', 'vernacularName', 'iucnStatus'
]].copy()

threatened_map = KeplerGl(height=700, width=1200)
threatened_map.add_data(data=threatened_map_df, name='Threatened Species Roadkill')
threatened_map
